# Diabetes Health Indicators: EDA + Modeling + Results

This end-to-end notebook combines exploratory data analysis, preprocessing, model training, evaluation, and interpretation in one reproducible workflow.

## Learning objectives
1. Understand the dataset and its data quality.
2. Explore diabetes prevalence and important health patterns.
3. Build leakage-safe preprocessing and classification pipelines.
4. Compare models using metrics appropriate for class imbalance.
5. Interpret the final results without treating association as causation.


## 1. Imports and configuration

We use pandas for data handling, seaborn/matplotlib for visualization, and scikit-learn for modeling. A fixed random seed makes the train/test split reproducible.

In [ ]:
import os
import urllib.request
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, confusion_matrix,
                             RocCurveDisplay)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 100)

## 2. Load the data

Place `diabetes_binary_health_indicators_BRFSS2015.csv` in a `data/` folder. If it is not found, the notebook attempts to download it from the reference repository. For production work, verify the source and dataset version before using the download option.

In [ ]:
DATA_PATH = 'data/diabetes_binary_health_indicators_BRFSS2015.csv'
DATA_URL = 'https://raw.githubusercontent.com/eviekenna/FinalProjectST558/main/diabetes_binary_health_indicators_BRFSS2015.csv'

if not os.path.exists(DATA_PATH):
    os.makedirs('data', exist_ok=True)
    try:
        urllib.request.urlretrieve(DATA_URL, DATA_PATH)
        print(f'Dataset downloaded to {DATA_PATH}')
    except Exception as exc:
        raise FileNotFoundError(
            f'Place the CSV at {DATA_PATH} or update DATA_URL. Original error: {exc}'
        )

df = pd.read_csv(DATA_PATH)
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
display(df.head())

## 3. Data audit

Before looking for relationships, check dimensions, data types, duplicates, missing values, and valid ranges. This prevents incorrect conclusions caused by data quality problems.

In [ ]:
display(df.info())
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_pct': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique(),
}).sort_values('missing_pct', ascending=False)
display(audit)
print('Duplicate rows:', df.duplicated().sum())
display(df.describe().T)

## 4. Target distribution

`Diabetes_binary` is the classification target. We inspect both counts and proportions because a majority-class baseline can make accuracy look better than the model's ability to identify the positive class.

In [ ]:
target_counts = df['Diabetes_binary'].value_counts().sort_index()
target_summary = pd.DataFrame({
    'count': target_counts,
    'proportion': target_counts / len(df)
})
display(target_summary)

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='Diabetes_binary', palette='Set2', ax=ax)
ax.set_title('Diabetes target distribution')
ax.set_xlabel('Target: 0 = no diabetes, 1 = prediabetes/diabetes')
ax.set_ylabel('Number of respondents')
plt.tight_layout()
plt.show()

majority_baseline = target_counts.max() / target_counts.sum()
print(f'Majority-class accuracy baseline: {majority_baseline:.3f}')

**Interpretation.** If class 0 is much more common, a model should not be selected using accuracy alone. Recall for the positive class, F1-score, and ROC-AUC provide a more informative view. In a healthcare application, the cost of false negatives should also be discussed with domain experts.

## 5. Feature distributions and health patterns

BMI, physical health, and mental health are numerical variables. Their distributions show skewness, unusual values, and potential differences between target groups.

In [ ]:
numeric_to_plot = [c for c in ['BMI', 'MentHlth', 'PhysHlth'] if c in df.columns]
fig, axes = plt.subplots(1, len(numeric_to_plot), figsize=(15, 4))
if len(numeric_to_plot) == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_to_plot):
    sns.histplot(data=df, x=col, bins=30, kde=True, ax=ax, color='#3274a1')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

if 'BMI' in df.columns:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x='Diabetes_binary', y='BMI', palette='Set2')
    plt.title('BMI by diabetes status')
    plt.xlabel('Diabetes status')
    plt.tight_layout()
    plt.show()

**Interpretation.** Differences in distributions are useful signals, but they are not causal evidence. Overlapping distributions also show why multiple features may be needed for prediction.

## 6. Binary risk-factor analysis

For binary health indicators, row-normalized tables answer: among people with or without a given condition, what percentage belong to each diabetes class? This is more informative than raw counts when group sizes differ.

In [ ]:
risk_features = [c for c in ['HighBP', 'HighChol', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'DiffWalk', 'Smoker'] if c in df.columns]
risk_results = []
for feature in risk_features:
    rates = df.groupby(feature)['Diabetes_binary'].mean()
    for level, rate in rates.items():
        risk_results.append({'feature': feature, 'level': level, 'diabetes_rate': rate})
risk_rates = pd.DataFrame(risk_results)
display(risk_rates.sort_values('diabetes_rate', ascending=False))

if risk_features:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()
    for ax, feature in zip(axes, risk_features):
        rate_table = df.groupby(feature)['Diabetes_binary'].mean().reset_index()
        sns.barplot(data=rate_table, x=feature, y='Diabetes_binary', ax=ax, color='#4c78a8')
        ax.set_title(f'Diabetes rate by {feature}')
        ax.set_ylabel('Mean target / rate')
        ax.tick_params(axis='x', rotation=25)
    for ax in axes[len(risk_features):]:
        ax.remove()
    plt.tight_layout()
    plt.show()

## 7. Modeling design

We now separate predictors and target. The split is stratified so the class proportions are preserved. All preprocessing is placed inside a pipeline to prevent train/test leakage.

The baseline models are: logistic regression for an interpretable linear benchmark, random forest for nonlinear relationships, and gradient boosting for a strong ensemble comparison.

In [ ]:
X = df.drop(columns='Diabetes_binary')
y = df['Diabetes_binary']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_features),
    ('categorical', categorical_pipe, categorical_features)
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=250, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}

pipelines = {
    name: Pipeline([('preprocessor', preprocessor), ('model', model)])
    for name, model in models.items()
}
print(f'Train rows: {len(X_train):,} | Test rows: {len(X_test):,}')

## 8. Cross-validation comparison

Five-fold stratified cross-validation estimates how consistently each pipeline performs on unseen folds. We report accuracy, precision, recall, F1, and ROC-AUC.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'accuracy': 'accuracy', 'precision': 'precision', 'recall': 'recall', 'f1': 'f1', 'roc_auc': 'roc_auc'}
cv_rows = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    row = {'model': name}
    for metric in scoring:
        row[f'{metric}_mean'] = scores[f'test_{metric}'].mean()
        row[f'{metric}_std'] = scores[f'test_{metric}'].std()
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values('roc_auc_mean', ascending=False)
display(cv_results.round(3))

## 9. Holdout test-set results

After comparing cross-validation results, fit each pipeline on the training data and evaluate once on the untouched test set. The test set approximates performance on future data.

In [ ]:
test_rows = []
fitted_models = {}
positive_label = 1

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted_models[name] = pipe
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, list(pipe.classes_).index(positive_label)]
    test_rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba)
    })

test_results = pd.DataFrame(test_rows).sort_values('roc_auc', ascending=False)
display(test_results.round(3))

best_name = test_results.iloc[0]['model']
best_model = fitted_models[best_name]
print(f'Best model by test ROC-AUC: {best_name}')
print(classification_report(y_test, best_model.predict(X_test), target_names=['No diabetes', 'Prediabetes/diabetes'], zero_division=0))

## 10. Confusion matrix and ROC curves

The confusion matrix shows the types of errors. The ROC curve summarizes the trade-off between true-positive and false-positive rates across probability thresholds.

In [ ]:
best_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, best_pred, labels=[0, 1])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No diabetes', 'Positive'],
            yticklabels=['No diabetes', 'Positive'])
plt.title(f'Confusion matrix: {best_name}')
plt.xlabel('Predicted label')
plt.ylabel('Actual label')
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
for name, pipe in fitted_models.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=ax)
ax.set_title('ROC curves on the holdout test set')
plt.show()

## 11. Model interpretation

For tree models, feature importance gives a useful first view of which transformed features contribute to predictions. It is not a causal effect and should be supplemented with permutation importance or SHAP for deeper interpretation.

In [ ]:
if 'Random Forest' in fitted_models:
    rf_pipe = fitted_models['Random Forest']
    rf = rf_pipe.named_steps['model']
    names = rf_pipe.named_steps['preprocessor'].get_feature_names_out()
    importance = pd.DataFrame({'feature': names, 'importance': rf.feature_importances_})
    importance = importance.sort_values('importance', ascending=False).head(20)
    display(importance)
    plt.figure(figsize=(10, 7))
    sns.barplot(data=importance.sort_values('importance'), x='importance', y='feature', color='#4c78a8')
    plt.title('Top Random Forest feature importances')
    plt.tight_layout()
    plt.show()

## 12. Final conclusions

### EDA findings
- The target distribution should be checked before selecting evaluation metrics.
- BMI, high blood pressure, high cholesterol, age, and general health indicators commonly show useful associations with diabetes status.
- Associations in a survey dataset do not establish that a feature causes diabetes.

### Modeling findings
- Pipelines keep imputation, scaling, and encoding inside the training process, reducing leakage risk.
- The best model should be selected using the metric that matches the application. ROC-AUC is useful for ranking, while recall may be prioritized when missing positive cases is especially costly.
- Holdout results are estimates, not guarantees of clinical performance. External validation, calibration, fairness analysis, and domain review are required before any real-world use.

### Suggested next steps
1. Tune hyperparameters using cross-validation.
2. Evaluate probability calibration and threshold selection.
3. Add permutation importance or SHAP explanations.
4. Check performance across age, sex, and socioeconomic groups.
5. Validate on a separate dataset collected from a relevant population.